In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

np.random.seed(1234)  # Semente para reprodutibilidade


In [ ]:
# Funções auxiliares

def calculateErrorSurface(X, y):
    N = len(y)
    M = 200
    a0 = np.linspace(-10.0, 14.0, M)
    a1 = np.linspace(-10.0, 14.0, M)
    A0, A1 = np.meshgrid(a0, a1)
    J = np.zeros((M,M))

    for i in range(M):
        for j in range(M):
            yhat = A0[i,j] + A1[i,j]*X
            J[i,j] = (1.0/N)*np.sum(np.square(y - yhat))
    return J, A0, A1

def calculateClosedFormSolution(X, y):
    N = len(y)
    a_opt = np.linalg.pinv(X.T.dot(X)).dot(X.T.dot(y))
    yhat = X.dot(a_opt)
    Joptimum = (1.0/N)*np.sum(np.square(y - yhat))
    return Joptimum, a_opt

def batchGradientDescent(X, y, alpha=0.1, n_iterations=1000):
    N = len(y)
    Jgd = np.zeros(n_iterations+1)
    a = np.array([-10.0, -10.0]).reshape(2, 1)
    a_hist = np.zeros((2, n_iterations+1))
    a_hist[:, 0] = a.ravel()

    yhat = X.dot(a)
    Jgd[0] = (1.0/N)*np.sum((y - yhat)**2)

    error = 1
    iteration = 0
    while error > 1e-12 and iteration < n_iterations:
        h = X.dot(a)
        gradients = -(2.0/N)*X.T.dot(y - h)
        a = a - alpha * gradients
        a_hist[:, iteration+1] = a.ravel()
        yhat = X.dot(a)
        Jgd[iteration+1] = (1.0/N)*np.sum((y - yhat)**2)
        error = abs(Jgd[iteration] - Jgd[iteration+1])
        iteration += 1
    return a, a_hist, Jgd, iteration


In [ ]:
# Geração de dados
N = 1000
x = 2 * np.random.rand(N, 1) - 1
w = np.random.randn(N, 1)
y = 2.0 + 2.0*x
y_noisy = y + w

plt.scatter(x, y_noisy, label='Função observada')
plt.plot(x, y, color='black', label='Modelo real')
plt.legend()
plt.grid()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados de treino")
plt.show()


In [ ]:
# Cálculo da superfície de erro
J, A0, A1 = calculateErrorSurface(x, y_noisy)

fig = plt.figure(figsize=(10, 4))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(A0, A1, J, cmap=cm.coolwarm)
ax1.set_title("Superfície do erro")
ax1.set_xlabel("a0")
ax1.set_ylabel("a1")

ax2 = fig.add_subplot(1, 2, 2)
cp = ax2.contour(A0, A1, J)
ax2.clabel(cp)
ax2.set_title("Contorno da função de erro")
ax2.set_xlabel("a0")
ax2.set_ylabel("a1")

plt.tight_layout()
plt.show()


In [ ]:
# Treinamento com Equação Normal
X = np.c_[np.ones((N, 1)), x]
Joptimum, a_opt = calculateClosedFormSolution(X, y_noisy)

print("Solução ótima (Equação normal)")
print(f"a0 = {a_opt[0, 0]:.4f}")
print(f"a1 = {a_opt[1, 0]:.4f}")
print(f"Erro ótimo = {Joptimum:.6f}")


In [ ]:
# Treinamento com Gradiente Descendente
alpha = 0.1
n_iterations = 1000
a, a_hist, Jgd, iteration = batchGradientDescent(X, y_noisy, alpha, n_iterations)

print("Solução via Gradiente Descendente")
print(f"a0 = {a[0, 0]:.4f}")
print(f"a1 = {a[1, 0]:.4f}")
print(f"Erro GD = {Jgd[iteration]:.6f}")


In [ ]:
# Curva de convergência do erro
plt.plot(np.arange(0, iteration), Jgd[:iteration])
plt.yscale("log")
plt.xlabel("Iteração")
plt.ylabel("Erro quadrático médio (MSE)")
plt.title("Convergência do Gradiente Descendente")
plt.grid()
plt.show()


In [ ]:
# Contorno da função de erro com trajetória do GD
plt.contour(A0, A1, J)
plt.plot(a_opt[0], a_opt[1], 'r*', label='Solução ótima')
plt.plot(a_hist[0, :iteration], a_hist[1, :iteration], 'kx', label='GD')
plt.xlabel("a0")
plt.ylabel("a1")
plt.legend()
plt.title("Caminho do Gradiente Descendente")
plt.grid()
plt.show()
